# Multi-site Compilation Builder

Select eligible site-level CSV files, harmonize them against CamCAN, and export compilation tables.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.clinical_combat.robust.robust_utils import get_metrics
from src.clinical_combat.robust.robust_harmonization import fit_script, apply_script

ROBUST_MODE = "HC"

RAW_SITES_DIR = Path("DATA") / "raw"
RAW_CAMCAN_DIR = Path("DATA") / "raw_camcan"
PROCESSED_ROOT = Path("DATA") / "processed"
CLEAN_CAMCAN_DIR = PROCESSED_ROOT / "CamCAN_clean"

init_harmonization_methods = ['pairwise'] # Harmonization methods to be applied on the datasets. The "pairwise" method is the one used in the original paper.

MIN_HC_SUBJECTS = 8
MIN_SICK_SUBJECTS = 5


In [ ]:
def get_camcan_metric_path(metric, cleaned=False):
    suffix = "clean" if cleaned else "raw"
    directory = CLEAN_CAMCAN_DIR if cleaned else RAW_CAMCAN_DIR
    return directory / f"CamCAN.{metric}.{suffix}.csv.gz"

def iter_site_files(metric):
    metric_dir = RAW_SITES_DIR / metric
    if not metric_dir.exists():
        return []
    return sorted(metric_dir.glob(f"*.{metric}.raw.csv.gz"))

def normalize_handedness_file(file_path):
    df = pd.read_csv(file_path)
    if "handedness" not in df.columns:
        return file_path
    updated = df["handedness"].replace(
    {False: 1, True: 2, "False": 1, "True": 2}).astype(int)
    if updated.equals(df["handedness"]):
        return file_path
    df["handedness"] = updated
    compression = "gzip" if str(file_path).endswith(".gz") else None
    df.to_csv(file_path, index=False, compression=compression)
    return file_path


def summarize_site_subjects(file_path):
    df = pd.read_csv(file_path, usecols=["sid", "disease"])
    unique = df.drop_duplicates()
    nb_total = unique["sid"].nunique()
    nb_hc = unique[unique["disease"] == "HC"]["sid"].nunique()
    nb_sick = nb_total - nb_hc
    return nb_total, nb_hc, nb_sick


def site_has_required_subjects(file_path, min_hc=MIN_HC_SUBJECTS, min_sick=MIN_SICK_SUBJECTS):
    nb_total, nb_hc, nb_sick = summarize_site_subjects(file_path)
    return (nb_hc >= min_hc and nb_sick >= min_sick), nb_total, nb_hc, nb_sick


def build_metric_compilation(metric):
    ref_file = get_camcan_metric_path(metric, cleaned=True)
    if not ref_file.exists():
        return None
    ref_file = normalize_handedness_file(ref_file)
    camcan_df = pd.read_csv(ref_file)
    camcan_df["source_site"] = "CamCAN"
    site_files = iter_site_files(metric)
    if not site_files:
        return None

    compiled_frames = []
    site_rows = []

    for site_file in site_files:
        site_file = normalize_handedness_file(site_file)
        site_name = site_file.name.split(f".{metric}.raw")[0]
        is_valid, nb_total, nb_hc, nb_sick = site_has_required_subjects(site_file)
        site_rows.append({
            "metric": metric,
            "site": site_name,
            "nb_total": nb_total,
            "nb_hc": nb_hc,
            "nb_sick": nb_sick,
            "selected": bool(is_valid),
            "source_path": str(site_file),
        })
        if not is_valid:
            continue

        model_dir = MODELS_DIR / metric
        model_dir.mkdir(parents=True, exist_ok=True)
        clean_dir = HARMONIZED_SITES_DIR / metric
        clean_dir.mkdir(parents=True, exist_ok=True)

        model_path = fit_script(
            str(site_file),
            str(ref_file),
            metric,
            HARMONIZATION_METHOD,
            ROBUST_MODE,
            str(model_dir),
        )
        harmonized_file = apply_script(
            str(site_file),
            model_path,
            metric,
            HARMONIZATION_METHOD,
            ROBUST_MODE,
            str(clean_dir),
        )
        harmonized_file = normalize_handedness_file(harmonized_file)
        harmonized_df = pd.read_csv(harmonized_file)
        harmonized_df["source_site"] = site_name
        compiled_frames.append(harmonized_df)

    summary_df = pd.DataFrame(site_rows)
    if not compiled_frames:
        return pd.DataFrame(), summary_df, camcan_df

    metric_df = pd.concat(compiled_frames, ignore_index=True)
    dst = COMPILATION_DIR / f"compilation.{metric}.csv.gz"
    metric_df.to_csv(dst, index=False, compression="gzip")

    metric_with_camcan = pd.concat([metric_df, camcan_df], ignore_index=True)
    dst_with_camcan = COMPILATION_DIR / f"compilation.{metric}.with_camcan.csv.gz"
    metric_with_camcan.to_csv(dst_with_camcan, index=False, compression="gzip")

    return metric_df, summary_df, camcan_df


In [ ]:
def build_full_compilation():    
    compilation_frames = []
    compilation_frames_with_camcan = []
    site_level_reports = []

    for metric in get_metrics():
        result = build_metric_compilation(metric)
        if result is None:
            continue
        metric_df, summary_df, camcan_df = result
        site_level_reports.append(summary_df)

        if not metric_df.empty:
            metric_df["metric_bundle"] = metric_df["metric"] + "_" + metric_df["bundle"]
            compilation_frames.append(metric_df)

            metric_with_camcan = pd.concat([metric_df, camcan_df], ignore_index=True)
            metric_with_camcan["metric_bundle"] = metric_with_camcan["metric"] + "_" + metric_with_camcan["bundle"]
            compilation_frames_with_camcan.append(metric_with_camcan)
        else:
            camcan_only = camcan_df.copy()
            camcan_only["metric_bundle"] = camcan_only["metric"] + "_" + camcan_only["bundle"]
            compilation_frames_with_camcan.append(camcan_only)

    if compilation_frames:
        compilation_all_metrics = pd.concat(compilation_frames, ignore_index=True)
        all_metrics_path = COMPILATION_DIR / "compilation.all_metrics.csv.gz"
        compilation_all_metrics.to_csv(all_metrics_path, index=False, compression="gzip")
    else:
        compilation_all_metrics = pd.DataFrame()

    if compilation_frames_with_camcan:
        compilation_all_metrics_with_camcan = pd.concat(compilation_frames_with_camcan, ignore_index=True)
        all_metrics_with_camcan_path = COMPILATION_DIR / "compilation.all_metrics.with_camcan.csv.gz"
        compilation_all_metrics_with_camcan.to_csv(
            all_metrics_with_camcan_path, index=False, compression="gzip"
        )
    else:
        compilation_all_metrics_with_camcan = pd.DataFrame()

    return pd.concat(site_level_reports, ignore_index=True) if site_level_reports else pd.DataFrame()


In [ ]:
for hm in init_harmonization_methods:
    HARMONIZATION_METHOD = hm
    COMPILATION_DIR = PROCESSED_ROOT / "compilation" / HARMONIZATION_METHOD / "all"
    MODELS_DIR = PROCESSED_ROOT / "models" / HARMONIZATION_METHOD
    HARMONIZED_SITES_DIR = PROCESSED_ROOT / "harmonized_sites" / HARMONIZATION_METHOD

    for path in [PROCESSED_ROOT, CLEAN_CAMCAN_DIR, COMPILATION_DIR, MODELS_DIR, HARMONIZED_SITES_DIR]:
        path.mkdir(parents=True, exist_ok=True)
    site_summary_df =  build_full_compilation()
    print("Site selection summary (first rows):")
    display(site_summary_df.head(20))
